# WiMANS on Kaggle - people counting, aimed at accuracy and ESP32 deployment (v3)

**Data:** WiMANS (Huang et al.), Intel 5300, 3 x 3 antennas, 30 subcarriers, ~1 kHz, 3-s samples,
**0-5 people**, 3 environments x 2 bands (2.4 / 5 GHz). Labels in `annotation.csv`
(`label`, `environment`, `wifi_band`, `number_of_users`, `user_<k>_location`, `user_<k>_activity`).
This notebook uses `wifi_csi/amp/<label>.npy` (amplitude, shape T x 3 x 3 x 30).

**What earlier runs showed:** raw 2.4 GHz amplitudes are white noise packet to packet (lag-1 autocorrelation
~0.00-0.07 at every count), so only **5 GHz** is used from Phase 1 on. On 5 GHz, empty rooms are found ~100 %
of the time, but 3 / 4 / 5 people blur together: total "wobble" saturates after ~2 people.

| Part | What it does |
|---|---|
| Phase 0 | labels, files, shapes, and the per-band signal probe (both bands, to keep the 2.4 GHz evidence) |
| Phase 1 | 5 GHz samples -> 9 link **views**: 30 subcarriers -> 52 tones, 50 Hz, normalised, 10 base count cues |
| Phase 2a | 9 new cues per link aimed at **how many separate movers** (PCA spread, Doppler spread, burstiness, tone spread) + 2 cross-link cues |
| Phase 2b | LightGBM on 3 feature sets, tiny CNN on one link and on all 9 links pooled |
| Phase 2c | deployment: int8 weights, parameters, multiply-adds, ONNX export |

**Targets:** exact count 0-5 and the coarser **0 / 1 / 2 / 3+** classes (`acc4`, macro `f1_4`).

**Test settings** (always a room the model was not trained on):

| Setting | What the installer does in the new room |
|---|---|
| `zero_shot` | nothing |
| `cal_empty` | records 30 s of the empty room |
| `cal_all` | records 30 s for each people count 0-5 |

Calibration data comes from one session per (room, count); that whole session is removed from the test set
in **every** setting, so all three settings are scored on exactly the same samples.

WiMANS samples are 3 s long, so the window here is the whole 3-s sample (150 bins), not 5 s.

**Kaggle settings:** attach your WiMANS dataset (Add Input), turn on **GPU** (T4 or P100) and Internet is not needed.
Outputs go to `/kaggle/working/` (`shared_format/wimans/`, `results/wimans/`). After *Save Version*,
the output can be attached to later notebooks as a dataset.

In [ ]:
# 1. Setup and paths
import copy
import glob
import json
import os
import time
from multiprocessing import get_context

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as Fnn
from sklearn.metrics import f1_score

# Locate dataset folders under /kaggle/input (works for any dataset slug / nesting)
SKIP_DIRS = {"amp", "mat", "video"}            # huge leaf folders: never walk into them


def find_dirs(test, root="/kaggle/input", max_depth=6):
    """Return folders for which test(folder, file_names) is True (shallow walk)."""
    hits = []
    base = root.rstrip("/").count("/")
    for folder, subdirs, files in os.walk(root):
        if test(folder, files):
            hits.append(folder)
        if folder.count("/") - base >= max_depth:
            subdirs[:] = []
        subdirs[:] = [s for s in subdirs if s not in SKIP_DIRS and not s.startswith("sub_")]
    return sorted(hits)


print("attached datasets:", sorted(os.listdir("/kaggle/input")))

roots = find_dirs(lambda folder, files: "annotation.csv" in files)
assert roots, "annotation.csv not found - is the WiMANS dataset attached?"
WIMANS_ROOT = roots[0]
ANNOT = os.path.join(WIMANS_ROOT, "annotation.csv")
AMP_DIR = os.path.join(WIMANS_ROOT, "wifi_csi", "amp")
if not os.path.isdir(AMP_DIR):
    hits = glob.glob(os.path.join(WIMANS_ROOT, "**", "amp"), recursive=True)
    AMP_DIR = hits[0]

OUT_ROOT = "/kaggle/working/shared_format"
SHARED = os.path.join(OUT_ROOT, "wimans")
RESULTS = "/kaggle/working/results/wimans"
os.makedirs(SHARED, exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)

# ---- shared-format settings (identical to the local-dataset notebooks) ----
FS = 50              # common sample rate (Hz)
MAX_GAP_BINS = 3     # gaps up to 60 ms are interpolated
MAX_MISSING = 0.20   # samples/windows with more missing bins are dropped
N_TONES = 52
WIN_S = 3.0          # one WiMANS sample = one window
HOP_S = 3.0
WIMANS_RATE = 1000   # nominal packets/s (amp files carry no timestamps)
BANDS_USED = ["5"]   # 2.4 GHz is white noise in WiMANS (see the Phase 0 signal probe)
N_BINS = int(WIN_S * FS)
N_WORKERS = os.cpu_count()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0
print("WiMANS root:", WIMANS_ROOT)
print("amp dir    :", AMP_DIR, "  files:", len(os.listdir(AMP_DIR)))
print("device     :", DEVICE, "  cpus:", N_WORKERS)

In [ ]:
# 2. Phase 0 - labels
ann = pd.read_csv(ANNOT, dtype=str)
print(ann.shape)
print(list(ann.columns))

ann["count"] = ann["number_of_users"].astype(int)
ann["session"] = ann["label"].str.split("_").str[1]          # act_<session>_<sample>
ann["has_file"] = [os.path.exists(os.path.join(AMP_DIR, l + ".npy")) for l in ann["label"]]
print("annotation rows with an amp file:", ann["has_file"].sum(), "/", len(ann))

display(pd.crosstab([ann["environment"], ann["wifi_band"]], ann["count"], margins=True))

# how many sessions per environment/band, and are counts mixed inside a session?
per_session = ann.groupby("session").agg(env=("environment", "first"), band=("wifi_band", "first"),
                                         n=("label", "size"), counts=("count", "nunique"),
                                         envs=("environment", "nunique"), bands=("wifi_band", "nunique"))
print("sessions:", len(per_session), "| sessions spanning >1 env:", (per_session["envs"] > 1).sum(),
      "| >1 band:", (per_session["bands"] > 1).sum(), "| with >1 count:", (per_session["counts"] > 1).sum())
display(per_session.groupby(["env", "band"]).agg(sessions=("n", "size"), samples=("n", "sum")))

act_cols = [c for c in ann.columns if c.endswith("_activity")]
print("activities:", sorted(set(ann[act_cols].stack().dropna()) - {"nan"}))

In [ ]:
# 3. Phase 0 - array shapes, value ranges, and time correlation per environment/band
rng = np.random.default_rng(SEED)
probe = (ann[ann["has_file"]].sample(frac=1, random_state=SEED)
         .groupby(["environment", "wifi_band", "count"]).head(8))


def probe_file(label):
    A = np.load(os.path.join(AMP_DIR, label + ".npy")).astype(np.float32)
    x = A.reshape(A.shape[0], -1)
    x = x - x.mean(0)
    num = (x[:-1] * x[1:]).sum(0)
    den = np.sqrt((x[:-1] ** 2).sum(0) * (x[1:] ** 2).sum(0)) + 1e-9
    return {"label": label, "shape": str(A.shape), "T": A.shape[0],
            "min": float(A.min()), "max": float(A.max()), "zeros": int((A == 0).sum()),
            "lag1_autocorr": float(np.median(num / den))}


with get_context("fork").Pool(N_WORKERS) as pool:
    sig = pd.DataFrame(pool.map(probe_file, probe["label"].tolist()))
sig = sig.merge(ann[["label", "environment", "wifi_band", "count", "session"]], on="label")

print("shapes seen:", sig["shape"].value_counts().head().to_dict())
print("T range:", sig["T"].min(), "-", sig["T"].max(), "| max value:", sig["max"].max())
display(sig.groupby(["environment", "wifi_band"])["lag1_autocorr"].describe().round(3))
display(sig.groupby("session")["lag1_autocorr"].median().round(2).sort_values().to_frame().T)
sig.to_csv(os.path.join(RESULTS, "phase0_signal_probe.csv"), index=False)

In [ ]:
# 3. Shared-format helpers: sub-bands, 50 Hz grid, normalisation, count cues

# ---------------------------------------------------------------
# 20 MHz sub-bands of an 80 MHz channel (256-point FFT, natural order)
# ---------------------------------------------------------------
# Sub-band s covers tone indices -128+64s ... -65+64s. We keep 52 tones,
# skipping 6 at each edge. This automatically avoids DC (bins 0, 1, 255)
# and the guard band (bins 123-133) that carry no channel data.
def subband_bins(s):
    start = -128 + 64 * s
    tones = np.arange(start + 6, start + 6 + N_TONES)
    return np.mod(tones, 256)


SUBBANDS = [subband_bins(s) for s in range(4)]


# ---------------------------------------------------------------
# Normalisation
# ---------------------------------------------------------------
def gain_normalise(A):
    """Per packet: divide by the median over tones (removes AGC jumps)."""
    return A / (np.median(A, axis=1, keepdims=True) + 1e-6)


def recording_normalise(X):
    """Per recording + receiver: each tone relative to its mean -> fluctuation around 0."""
    mu = np.nanmean(X, axis=0, keepdims=True)
    dead = mu < 1e-3 * np.nanmedian(mu)                    # null tones, if any
    Y = X / (mu + 1e-6) - 1.0
    Y[:, dead[0]] = 0.0
    return Y


# ---------------------------------------------------------------
# Resampling onto the 50 Hz grid
# ---------------------------------------------------------------
def resample_to_grid(t, A):
    """t: packet times (s, sorted), A: (packets, tones).
    Returns X (bins, tones) with NaN rows for empty bins, and the valid mask."""
    idx = np.floor((t - t[0]) * FS).astype(np.int64)
    n_bins = int(idx[-1]) + 1

    uniq, first = np.unique(idx, return_index=True)        # idx is sorted
    sums = np.add.reduceat(A.astype(np.float64), first, axis=0)
    counts = np.diff(np.append(first, len(idx)))

    X = np.full((n_bins, A.shape[1]), np.nan, dtype=np.float32)
    X[uniq] = sums / counts[:, None]
    valid = np.zeros(n_bins, dtype=bool)
    valid[uniq] = True
    return X, valid


def fill_short_gaps(X, valid):
    """Linear interpolation over gaps of at most MAX_GAP_BINS; longer gaps stay NaN."""
    X, valid = X.copy(), valid.copy()
    missing = np.flatnonzero(~valid)
    if len(missing) == 0:
        return X, valid

    runs = np.split(missing, np.flatnonzero(np.diff(missing) > 1) + 1)
    for run in runs:
        left, right = run[0] - 1, run[-1] + 1
        if len(run) <= MAX_GAP_BINS and left >= 0 and right < len(X):
            w = ((run - left) / (right - left))[:, None]
            X[run] = X[left] * (1 - w) + X[right] * w
            valid[run] = True
    return X, valid


def to_shared_series(t, A):
    """Raw amplitude of ONE view -> normalised 50 Hz series (bins, 52) + valid mask."""
    A = gain_normalise(A.astype(np.float32))
    X, valid = resample_to_grid(t, A)
    X, valid = fill_short_gaps(X, valid)
    return recording_normalise(X), valid


# ---------------------------------------------------------------
# Count cues per window
# ---------------------------------------------------------------
FEATURE_NAMES = [
    "cv_median",       # median over tones of the fluctuation std
    "cv_p90",          # 90th percentile of the same
    "cv_p10",          # 10th percentile of the same
    "diff_energy",     # mean |x(t) - x(t-1)|  (motion energy)
    "pc1_frac",        # share of variance in the first principal component
    "n_pc_90",         # components needed for 90 % of the variance
    "tone_corr",       # mean correlation between tones
    "low_freq_frac",   # temporal power below 2 Hz (slow body motion)
    "high_freq_frac",  # temporal power above 5 Hz
    "mean_std",        # std of the tone-averaged series
]


def window_features(W):
    """W: (bins, 52) normalised window. Missing bins count as 'no change' (0)."""
    W = np.nan_to_num(W.astype(np.float32))
    sd = W.std(axis=0)
    diff_energy = np.abs(np.diff(W, axis=0)).mean()

    Xc = W - W.mean(axis=0)
    s = np.linalg.svd(Xc, compute_uv=False) ** 2
    frac = s / (s.sum() + 1e-12)
    n90 = np.searchsorted(np.cumsum(frac), 0.90) + 1

    with np.errstate(invalid="ignore", divide="ignore"):
        C = np.corrcoef(Xc.T)
    tone_corr = np.nanmean(C[np.triu_indices_from(C, k=1)])

    x = Xc.mean(axis=1)
    P = np.abs(np.fft.rfft(x)) ** 2
    f = np.fft.rfftfreq(len(x), 1.0 / FS)
    p_total = P[f > 0].sum() + 1e-12
    low = P[(f > 0) & (f < 2.0)].sum() / p_total
    high = P[f > 5.0].sum() / p_total

    return np.array([np.median(sd), np.percentile(sd, 90), np.percentile(sd, 10),
                     diff_energy, frac[0], n90, tone_corr, low, high, x.std()],
                    dtype=np.float32)


# ---------------------------------------------------------------
# Windowing + saving one recording
# ---------------------------------------------------------------
def save_recording(rec):
    """rec: dict with series (views, bins, 52), valid (bins,), view_names, metadata."""
    S, valid = rec["series"], rec["valid"]
    win, hop = int(WIN_S * FS), int(HOP_S * FS)

    starts = [a for a in range(0, S.shape[1] - win + 1, hop)
              if valid[a:a + win].mean() >= 1 - MAX_MISSING]

    F = np.zeros((S.shape[0], len(starts), len(FEATURE_NAMES)), dtype=np.float32)
    for v in range(S.shape[0]):
        for j, a in enumerate(starts):
            F[v, j] = window_features(S[v, a:a + win])

    meta = {k: rec[k] for k in ["dataset", "recording", "group", "setup", "receiver", "count"]}
    meta["seconds"] = round(S.shape[1] / FS, 1)
    meta["missing_frac"] = round(float(1 - valid.mean()), 4)
    meta["n_windows"] = len(starts)

    out_dir = os.path.join(OUT_ROOT, rec["dataset"])
    os.makedirs(out_dir, exist_ok=True)
    np.savez_compressed(os.path.join(out_dir, rec["recording"] + ".npz"),
                        series=S.astype(np.float16), valid=valid,
                        starts=np.array(starts, dtype=np.int32), feats=F,
                        view_names=np.array(rec["view_names"]), meta=json.dumps(meta))
    return meta


def already_done(dataset, recording):
    return os.path.exists(os.path.join(OUT_ROOT, dataset, recording + ".npz"))


print("sub-band bins:", [(int(b[0]), int(b[-1])) for b in SUBBANDS])


# Map any number of subcarriers onto the 52-tone grid (linear interpolation over frequency)
def tone_interp_matrix(n_src, n_dst=N_TONES):
    src = np.linspace(0.0, 1.0, n_src)
    dst = np.linspace(0.0, 1.0, n_dst)
    W = np.zeros((n_dst, n_src), dtype=np.float32)
    for i, x in enumerate(dst):
        j = min(np.searchsorted(src, x, side="right") - 1, n_src - 2)
        w = (x - src[j]) / (src[j + 1] - src[j])
        W[i, j] = 1.0 - w
        W[i, j + 1] = w
    return W

In [ ]:
# 4. Phase 1 - convert every 5 GHz sample to the shared format (9 views x 150 bins x 52 tones)
W30 = tone_interp_matrix(30)
labels = ann.loc[ann["has_file"] & ann["wifi_band"].isin(BANDS_USED), "label"].tolist()
print("samples to convert:", len(labels), "bands:", BANDS_USED)


def convert_sample(label):
    A = np.load(os.path.join(AMP_DIR, label + ".npy")).astype(np.float32)   # (T, 3, 3, 30)
    t = np.arange(A.shape[0]) / WIMANS_RATE
    series = np.full((9, N_BINS, N_TONES), np.nan, dtype=np.float32)
    valid = np.zeros(N_BINS, dtype=bool)
    feats = np.zeros((9, len(FEATURE_NAMES)), dtype=np.float32)

    for v in range(9):
        a, b = divmod(v, 3)
        X, ok = to_shared_series(t, A[:, a, b, :] @ W30.T)
        n = min(len(X), N_BINS)
        series[v, :n] = X[:n]
        valid[:n] = ok[:n]
        feats[v] = window_features(series[v])
    return label, series.astype(np.float16), valid, feats


N = len(labels)
SERIES = np.zeros((N, 9, N_BINS, N_TONES), dtype=np.float16)       # ~0.8 GB for 5 GHz
VALID = np.zeros((N, N_BINS), dtype=bool)
FEATS = np.zeros((N, 9, len(FEATURE_NAMES)), dtype=np.float32)
pos = {l: i for i, l in enumerate(labels)}

t0 = time.time()
with get_context("fork").Pool(N_WORKERS) as pool:
    for k, (label, s, ok, f) in enumerate(pool.imap_unordered(convert_sample, labels, chunksize=16)):
        i = pos[label]
        SERIES[i], VALID[i], FEATS[i] = s, ok, f
        if k % 1000 == 0:
            print(f"{k:6d} / {N}   [{time.time() - t0:.0f} s]", flush=True)

meta = ann.set_index("label").loc[labels].reset_index()
meta["valid_frac"] = VALID.mean(1)
np.save(os.path.join(SHARED, "series.npy"), SERIES)
np.save(os.path.join(SHARED, "feats.npy"), FEATS)
meta.to_parquet(os.path.join(SHARED, "meta.parquet"), index=False)
with open(os.path.join(SHARED, "format.json"), "w") as f:
    json.dump({"fs_hz": FS, "win_s": WIN_S, "n_tones": N_TONES, "views": "tx*3+rx (9 links)",
               "tone_mapping": "30 Intel-5300 subcarriers linearly interpolated to 52",
               "feature_names": FEATURE_NAMES}, f, indent=1)
print(f"done in {time.time() - t0:.0f} s; samples with >= 80 % valid bins:",
      int((meta["valid_frac"] >= 1 - MAX_MISSING).sum()), "/", N)

In [ ]:
# 5. Sanity check: does the main count cue rise with people in every environment/band?
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
cue = pd.DataFrame({"count": meta["count"], "env": meta["environment"], "band": meta["wifi_band"],
                    "cv_median": FEATS[:, :, 0].mean(1), "diff_energy": FEATS[:, :, 3].mean(1)})
for ax, col in zip(axes, ["cv_median", "diff_energy"]):
    for (env, band), g in cue.groupby(["env", "band"]):
        m = g.groupby("count")[col].median()
        ax.plot(m.index, m.values, marker="o", label=f"{env}/{band}")
    ax.set_xlabel("people")
    ax.set_ylabel(col + " (median, mean over 9 links)")
    ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(os.path.join(RESULTS, "cues_vs_count.png"), dpi=120)
plt.show()

display(cue.groupby(["env", "band"]).apply(
    lambda g: g["cv_median"].corr(g["count"], method="spearman")).rename("spearman").round(3))

In [ ]:
# 6. Phase 2a - extra cues for "how many separate movers" (all cheap enough for an ESP32)
# Extra cues for "how many separate movers" (all cheap enough for an ESP32)
EXTRA_NAMES = [
    "pc2_frac",          # share of variance in the 2nd principal component
    "pc3_frac",          # ... and in the 3rd (several movers spread variance over components)
    "eig_entropy",       # entropy of the PCA spectrum = effective number of independent motions
    "doppler_centroid",  # mean temporal frequency of the motion (Hz)
    "doppler_spread",    # spread of the temporal frequencies (Hz)
    "psd_entropy",       # flatness of the temporal spectrum (0..1)
    "activity_cv",       # burstiness of motion over 0.2-s segments (one mover: bursty, many: steady)
    "tone_energy_cv",    # how unevenly motion is spread over the 52 tones
    "half_band_corr",    # correlation of motion in the lower vs upper 26 tones
]
LINK_NAMES = [
    "link_env_corr",     # mean correlation of the 9 links' motion envelopes (needs several receivers)
    "link_energy_cv",    # spread of motion energy over the 9 links
]


def extra_features(W):
    """W: (bins, 52) normalised window -> (extra cues, motion envelope over time)."""
    W = np.nan_to_num(W.astype(np.float32))
    Xc = W - W.mean(axis=0)

    # spread of the motion over principal components
    s = np.linalg.svd(Xc, compute_uv=False) ** 2
    frac = s / (s.sum() + 1e-12)
    eig_entropy = -(frac * np.log(frac + 1e-12)).sum()

    # temporal (Doppler) spectrum summed over tones, DC removed
    P = (np.abs(np.fft.rfft(Xc, axis=0)) ** 2).sum(axis=1)[1:]
    f = np.fft.rfftfreq(len(W), 1.0 / FS)[1:]
    p = P / (P.sum() + 1e-12)
    centroid = (p * f).sum()
    spread = np.sqrt((p * (f - centroid) ** 2).sum())
    psd_entropy = -(p * np.log(p + 1e-12)).sum() / np.log(len(p))

    # motion = absolute change between bins
    d = np.abs(np.diff(W, axis=0))
    env = d.mean(axis=1)
    seg = env[:len(env) // 10 * 10].reshape(-1, 10).mean(axis=1)
    activity_cv = seg.std() / (seg.mean() + 1e-9)
    tone_e = d.mean(axis=0)
    tone_energy_cv = tone_e.std() / (tone_e.mean() + 1e-9)

    half = W.shape[1] // 2
    lo, hi = d[:, :half].mean(axis=1), d[:, half:].mean(axis=1)
    half_band_corr = np.corrcoef(lo, hi)[0, 1] if lo.std() > 0 and hi.std() > 0 else 0.0

    cues = np.array([frac[1], frac[2], eig_entropy, centroid, spread, psd_entropy,
                     activity_cv, tone_energy_cv, half_band_corr], dtype=np.float32)
    return cues, env


def link_cues(envs):
    """Cross-link cues from the motion envelopes of all links of one sample."""
    E = np.array(envs)
    with np.errstate(invalid="ignore", divide="ignore"):
        C = np.corrcoef(E)
    energy = E.mean(axis=1)
    return np.array([np.nanmean(C[np.triu_indices(len(E), k=1)]),
                     energy.std() / (energy.mean() + 1e-9)], dtype=np.float32)


def sample_extra(i):
    cues, envs = [], []
    for v in range(9):
        c, e = extra_features(SERIES[i, v])
        cues.append(c)
        envs.append(e)
    return i, np.array(cues), link_cues(envs)


EXTRA = np.zeros((N, 9, len(EXTRA_NAMES)), dtype=np.float32)
LINK = np.zeros((N, len(LINK_NAMES)), dtype=np.float32)
t0 = time.time()
with get_context("fork").Pool(N_WORKERS) as pool:
    for i, cues, link in pool.imap_unordered(sample_extra, range(N), chunksize=64):
        EXTRA[i], LINK[i] = cues, link
print(f"extra cues for {N} samples in {time.time() - t0:.0f} s")
np.save(os.path.join(SHARED, "extra_feats.npy"), EXTRA)
np.save(os.path.join(SHARED, "link_feats.npy"), LINK)

# how each cue moves with the people count, per room (median over samples, mean over links)
cue = pd.DataFrame(EXTRA.mean(axis=1), columns=EXTRA_NAMES)
cue[LINK_NAMES] = LINK
cue["count"], cue["env"] = meta["count"].values, meta["environment"].values
names = EXTRA_NAMES + LINK_NAMES
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, col in zip(axes.flat, names):
    for env, g in cue.groupby("env"):
        m = g.groupby("count")[col].median()
        ax.plot(m.index, m.values, marker="o", label=env)
    ax.set_title(col, fontsize=9)
    ax.set_xlabel("people")
axes.flat[0].legend(fontsize=7)
for ax in axes.flat[len(names):]:
    ax.axis("off")
fig.tight_layout()
fig.savefig(os.path.join(RESULTS, "extra_cues_vs_count.png"), dpi=110)
plt.show()

spearman = pd.DataFrame({env: [g[c].corr(g["count"], method="spearman") for c in names]
                         for env, g in cue.groupby("env")}, index=names).round(3)
spearman.to_csv(os.path.join(RESULTS, "extra_cue_spearman.csv"))
display(spearman)

In [ ]:
# 7. Phase 2 - feature table (one row per sample x link), calibration sessions, settings
keep = (meta["valid_frac"] >= 1 - MAX_MISSING).values
idx = np.flatnonzero(keep)
MAX_COUNT = int(meta["count"].max())

rows = pd.DataFrame({"sample": np.repeat(idx, 9), "view": np.tile(np.arange(9), len(idx))})
for c in ["count", "environment", "session"]:
    rows[c] = meta[c].values[rows["sample"]]
rows["count4"] = np.minimum(rows["count"], 3)
rows["sample_no"] = meta["label"].str.split("_").str[2].astype(int).values[rows["sample"]]
rows[FEATURE_NAMES] = np.nan_to_num(FEATS[idx].reshape(-1, len(FEATURE_NAMES)))
rows[EXTRA_NAMES] = np.nan_to_num(EXTRA[idx].reshape(-1, len(EXTRA_NAMES)))
rows[LINK_NAMES] = np.nan_to_num(np.repeat(LINK[idx], 9, axis=0))
rows["calib_group"] = rows["environment"]
ENVS = sorted(rows["environment"].unique())

FEATURE_SETS = {
    "base10": FEATURE_NAMES,                                    # the 10 cues of the earlier runs
    "single_link": FEATURE_NAMES + EXTRA_NAMES,                 # everything one ESP32 link can compute
    "multi_link": FEATURE_NAMES + EXTRA_NAMES + LINK_NAMES,     # + cues that need several receivers
}

# ---- install-time calibration: one session per (room, count), first 30 s of it ----
CAL_SAMPLES = 10                     # 10 samples x 3 s = 30 s per people count
sessions = rows.drop_duplicates("session")[["environment", "count", "session"]]
per_group = sessions.groupby(["environment", "count"])["session"].apply(sorted)
print("sessions per room and count:")
display(per_group.apply(len).unstack())

rng = np.random.default_rng(SEED)
CAL_SESSIONS = {s[rng.integers(len(s))] for s in per_group if len(s) >= 2}   # keep >= 1 session to test
rows["cal_session"] = rows["session"].isin(CAL_SESSIONS)
rank = rows.groupby("session")["sample_no"].rank(method="dense")
rows["cal_pick"] = rows["cal_session"] & (rank <= CAL_SAMPLES)

SETTINGS = {"zero_shot": None, "cal_empty": [0], "cal_all": list(range(MAX_COUNT + 1))}
VIEW0 = (rows["view"] == 0).values


def folds(setting):
    """Leave one room out. Calibration sessions are never tested; in the cal_* settings their
    first 30 s (only the listed counts) are added to the training data."""
    cal_counts = SETTINGS[setting]
    for env in ENVS:
        in_env = (rows["environment"] == env).values
        test = in_env & ~rows["cal_session"].values
        train = ~in_env
        if cal_counts is not None:
            train = train | (in_env & rows["cal_pick"].values & rows["count"].isin(cal_counts).values)
        yield env, train, test


print(len(idx), "samples,", len(rows), "rows,", len(CAL_SESSIONS), "calibration sessions,",
      "test samples:", int((~rows["cal_session"] & VIEW0).sum()))

In [ ]:
# Metrics and result collection
RESULT_ROWS, PER_COUNT_ROWS = [], []


def to4(c):
    """Map a people count to the 0 / 1 / 2 / 3+ classes."""
    return np.minimum(c, 3)


def score(y, pred, max_count):
    y = np.asarray(y, dtype=float)
    pred = np.asarray(pred, dtype=float)
    rounded = np.clip(np.rint(pred), 0, max_count)
    y4, r4 = to4(y), to4(rounded)
    labels4 = sorted(set(y4) | set(r4))                    # classes absent from both are not scored
    return {"n": len(y),
            "mae": float(np.mean(np.abs(pred - y))),
            "exact": float(np.mean(rounded == y)),
            "within1": float(np.mean(np.abs(rounded - y) <= 1)),
            "acc4": float(np.mean(r4 == y4)),
            "f1_4": float(f1_score(y4, r4, average="macro", labels=labels4, zero_division=0))}


def evaluate(setting, model, task, calib, rows, pred):
    """rows: one row per (sample, view) with columns sample, count, count4; pred aligned with it.
    Task 'clf4' predicts the 0/1/2/3+ class directly, so it is scored against count4."""
    target = "count4" if task == "clf4" else "count"
    max_count = 3 if task == "clf4" else MAX_COUNT

    d = rows[["sample", target]].rename(columns={target: "y"})
    d["pred"] = pred
    RESULT_ROWS.append({"setting": setting, "model": model, "task": task, "calib": calib,
                        "level": "single_view", **score(d["y"], d["pred"], max_count)})

    ens = d.groupby("sample").agg(y=("y", "first"), pred=("pred", "mean"))
    RESULT_ROWS.append({"setting": setting, "model": model, "task": task, "calib": calib,
                        "level": "view_ensemble", **score(ens["y"], ens["pred"], max_count)})

    rounded = np.clip(np.rint(ens["pred"].values), 0, max_count)
    for c in sorted(ens["y"].unique()):
        m = ens["y"].values == c
        PER_COUNT_ROWS.append({"setting": setting, "model": model, "task": task, "calib": calib,
                               "count": int(c), "n": int(m.sum()),
                               "recall": float(np.mean(rounded[m] == c)),
                               "mean_pred": float(ens["pred"].values[m].mean())})
    return ens

In [ ]:
# 9. Phase 2b - LightGBM: 3 feature sets x 3 settings x (6-class, 4-class, regression)
GBM_PARAMS = dict(n_estimators=300, learning_rate=0.05, num_leaves=15, min_child_samples=40,
                  subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                  random_state=SEED, verbose=-1)


def calibrate(df, feats):
    """z-score each feature within (room, link) using that room's own unlabelled samples."""
    out = df.copy()
    g = out.groupby(["calib_group", "view"])[feats]
    out[feats] = (out[feats] - g.transform("mean")) / (g.transform("std") + 1e-6)
    return out


IMPORTANCE = []
t0 = time.time()
for calib in ["none", "group_z"]:
    for fs_name, feats in FEATURE_SETS.items():
        d = calibrate(rows, feats) if calib == "group_z" else rows
        for setting in SETTINGS:
            preds = {k: np.full(len(d), np.nan) for k in ["clf", "clf4", "reg", "mean"]}
            for env, train, test in folds(setting):
                X_tr, X_te = d.loc[train, feats], d.loc[test, feats]
                y_tr = d.loc[train, "count"]

                preds["clf"][test] = lgb.LGBMClassifier(**GBM_PARAMS).fit(X_tr, y_tr).predict(X_te)
                preds["clf4"][test] = lgb.LGBMClassifier(**GBM_PARAMS).fit(
                    X_tr, d.loc[train, "count4"]).predict(X_te)
                reg = lgb.LGBMRegressor(**GBM_PARAMS).fit(X_tr, y_tr)
                preds["reg"][test] = reg.predict(X_te)
                preds["mean"][test] = y_tr.mean()
                if setting == "zero_shot":
                    IMPORTANCE.append(pd.Series(reg.booster_.feature_importance("gain"), index=feats,
                                                name=f"{fs_name}|{calib}|{env}"))

            tested = ~np.isnan(preds["reg"])
            for task in ["clf", "clf4", "reg"]:
                evaluate(setting, f"gbm_{fs_name}", task, calib, d[tested], preds[task][tested])
            if calib == "none" and fs_name == "base10":
                evaluate(setting, "mean_baseline", "reg", "none", d[tested], preds["mean"][tested])
        print(f"calib={calib:8s} features={fs_name:12s} [{time.time() - t0:.0f} s]", flush=True)

res = pd.DataFrame(RESULT_ROWS)
display(res[res["level"] == "view_ensemble"].sort_values("f1_4", ascending=False).head(15).round(3))

In [ ]:
# Tiny 1D-CNN (~35k parameters, fits the ESP32-S3 budget). Input (batch, 52 tones, time).
class TinyCNN(nn.Module):
    def __init__(self, n_tones=N_TONES, n_classes=MAX_COUNT + 1):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv1d(n_tones, 32, kernel_size=7, padding=3), nn.BatchNorm1d(32), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 48, kernel_size=5, padding=4, dilation=2), nn.BatchNorm1d(48), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(48, 64, kernel_size=5, padding=4, dilation=2), nn.BatchNorm1d(64), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Dropout(0.2),
        )
        self.cls_head = nn.Linear(64, n_classes)
        self.reg_head = nn.Linear(64, 1)

    def forward(self, x):
        h = self.body(x)
        return self.cls_head(h), self.reg_head(h).squeeze(1)


class MultiLinkCNN(nn.Module):
    """Several receivers: the same TinyCNN encoder runs on every link, the link features are
    averaged, then the same heads. Same parameter count; input (batch, links, 52 tones, time)."""

    def __init__(self):
        super().__init__()
        self.net = TinyCNN()

    def forward(self, x):
        b, n_links = x.shape[:2]
        h = self.net.body(x.flatten(0, 1)).view(b, n_links, -1).mean(1)
        return self.net.cls_head(h), self.net.reg_head(h).squeeze(1)


print("TinyCNN parameters:", sum(p.numel() for p in TinyCNN().parameters()))

In [ ]:
# 11. Phase 2b - tiny CNNs: one link (one ESP32) and all 9 links pooled (several receivers)
N_VIEWS = 9                    # links per sample
EPOCHS = 15
BATCH = 256
SAMPLES_PER_EPOCH = 40000      # link windows per epoch; the multi-link model sees the same total
LR = 2e-3
CAL_SHARE = 0.2                # share of each epoch drawn from the new room's calibration samples


def load_x(samples, views=None, train=False, rng=None):
    """(batch, 52, 150) for one link per row, or (batch, 9, 52, 150) for all links (views=None)."""
    x = SERIES[samples] if views is None else SERIES[samples, views]
    x = np.nan_to_num(x.astype(np.float32)).clip(-3, 3)
    if train:
        # light augmentation against room changes: overall scale, per-tone gain, time shift
        lead = x.shape[:-2]
        x = x * rng.uniform(0.85, 1.15, size=lead + (1, 1))
        x = x * np.exp(rng.normal(0.0, 0.1, size=lead + (1, x.shape[-1])))
        x = np.roll(x, int(rng.integers(x.shape[-2])), axis=-2)
    x = np.ascontiguousarray(np.swapaxes(x, -1, -2), dtype=np.float32)    # time last
    return torch.from_numpy(x).to(DEVICE)


def epoch_weights(train_df, cal_mask):
    """Balanced over people counts; the new room's calibration rows get CAL_SHARE of each epoch."""
    w = 1.0 / train_df["count"].map(train_df["count"].value_counts()).values
    if cal_mask.any():
        w_cal, w_rest = w[cal_mask].sum(), w[~cal_mask].sum()
        w[cal_mask] *= CAL_SHARE * w_rest / ((1 - CAL_SHARE) * w_cal)
    return w / w.sum()


def train_model(kind, train_df, cal_mask):
    torch.manual_seed(SEED)
    rng = np.random.default_rng(SEED)
    model = (TinyCNN() if kind == "single" else MultiLinkCNN()).to(DEVICE)
    model.seen = torch.zeros(MAX_COUNT + 1, dtype=torch.bool, device=DEVICE)
    model.seen[torch.tensor(sorted(train_df["count"].unique()), device=DEVICE)] = True

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    n = SAMPLES_PER_EPOCH if kind == "single" else SAMPLES_PER_EPOCH // N_VIEWS
    batch = BATCH if kind == "single" else BATCH // 8
    p = epoch_weights(train_df, cal_mask)

    for epoch in range(EPOCHS):
        model.train()
        ep = train_df.iloc[rng.choice(len(train_df), size=n, replace=True, p=p)]
        for b in range(0, n, batch):
            sub = ep.iloc[b:b + batch]
            if len(sub) < 2:                                   # BatchNorm needs > 1 sample
                continue
            views = sub["view"].values if kind == "single" else None
            y = torch.tensor(sub["count"].values, device=DEVICE)
            logits, reg = model(load_x(sub["sample"].values, views, train=True, rng=rng))
            loss = Fnn.cross_entropy(logits, y) + Fnn.huber_loss(reg, y.float())
            opt.zero_grad()
            loss.backward()
            opt.step()
        sched.step()
    return model


def predict(model, kind, df):
    model.eval()
    step = 2048 if kind == "single" else 256
    p_cls, p_reg = [], []
    with torch.no_grad():
        for b in range(0, len(df), step):
            sub = df.iloc[b:b + step]
            views = sub["view"].values if kind == "single" else None
            logits, reg = model(load_x(sub["sample"].values, views))
            p_cls.append(logits.masked_fill(~model.seen, -1e9).argmax(1).cpu().numpy())
            p_reg.append(reg.cpu().numpy())
    return np.concatenate(p_cls), np.concatenate(p_reg)


def int8_copy(model):
    """Weights rounded to int8 (symmetric, per output channel), as stored on the ESP32."""
    q = copy.deepcopy(model)
    with torch.no_grad():
        for m in q.modules():
            if isinstance(m, (nn.Conv1d, nn.Linear)):
                w = m.weight
                s = w.abs().amax(dim=tuple(range(1, w.dim())), keepdim=True) / 127 + 1e-12
                m.weight.copy_(torch.round(w / s) * s)
    return q


t0 = time.time()
for kind in ["single", "multi"]:
    for setting in SETTINGS:
        out = {(v, t): np.full(len(rows), np.nan) for v in ["fp32", "int8"] for t in ["clf", "reg"]}
        for env, train, test in folds(setting):
            if kind == "multi":                                # one row per sample
                train, test = train & VIEW0, test & VIEW0
            tr = rows[train]
            model = train_model(kind, tr, (tr["environment"] == env).values)
            for v, m in [("fp32", model), ("int8", int8_copy(model))]:
                out[(v, "clf")][test], out[(v, "reg")][test] = predict(m, kind, rows[test])
            print(f"cnn_{kind:6s} {setting:9s} left out {env:14s} [{time.time() - t0:.0f} s]", flush=True)

        tested = ~np.isnan(out[("fp32", "reg")])
        for (v, task), pred in out.items():
            name = f"cnn_{kind}" + ("_int8" if v == "int8" else "")
            evaluate(setting, name, task, "none", rows[tested], pred[tested])

res = pd.DataFrame(RESULT_ROWS)
display(res[res["model"].str.startswith("cnn") & (res["level"] == "view_ensemble")]
        .sort_values(["setting", "f1_4"], ascending=[True, False]).round(3))

In [ ]:
# 12. Phase 2c - deployment: final CNNs on all 5 GHz data, size, compute, ONNX export
def count_macs(model, x):
    """Multiply-adds for one inference (x has batch size 1)."""
    macs = []

    def hook(m, inp, out):
        if isinstance(m, nn.Conv1d):
            macs.append(out.numel() * (m.in_channels // m.groups) * m.kernel_size[0])
        else:
            macs.append(out.numel() * m.in_features)

    hooks = [m.register_forward_hook(hook) for m in model.modules() if isinstance(m, (nn.Conv1d, nn.Linear))]
    with torch.no_grad():
        model(x)
    for h in hooks:
        h.remove()
    return int(sum(macs))


def export_onnx(model, x, path):
    kwargs = dict(input_names=["csi"], output_names=["count_logits", "count_reg"], opset_version=17)
    try:
        torch.onnx.export(model, x, path, dynamo=False, **kwargs)
    except TypeError:                                          # older torch: no 'dynamo' argument
        torch.onnx.export(model, x, path, **kwargs)


DEPLOY = {}
for kind in ["single", "multi"]:
    tr = rows if kind == "single" else rows[VIEW0]
    model = train_model(kind, tr, np.zeros(len(tr), dtype=bool))
    model.eval()
    shape = (1, N_TONES, N_BINS) if kind == "single" else (1, N_VIEWS, N_TONES, N_BINS)
    x = torch.zeros(shape, device=DEVICE)

    n_params = sum(p.numel() for p in model.parameters())
    DEPLOY[kind] = {"input": list(shape[1:]), "window_s": WIN_S, "params": n_params,
                    "macs_per_window": count_macs(model, x),
                    "weights_fp32_kb": round(n_params * 4 / 1024, 1),
                    "weights_int8_kb": round(n_params / 1024, 1)}
    torch.save(model.state_dict(), os.path.join(RESULTS, f"tinycnn_{kind}_5ghz.pt"))
    try:
        export_onnx(model.cpu(), x.cpu(), os.path.join(RESULTS, f"tinycnn_{kind}_5ghz.onnx"))
        DEPLOY[kind]["onnx"] = "ok"
    except Exception as e:                                     # report, don't stop
        DEPLOY[kind]["onnx"] = repr(e)[:200]

with open(os.path.join(RESULTS, "deployment.json"), "w") as f:
    json.dump(DEPLOY, f, indent=1)
display(pd.DataFrame(DEPLOY).T)

In [ ]:
# 13. Save results, plots and a short report
results = pd.DataFrame(RESULT_ROWS)
per_count = pd.DataFrame(PER_COUNT_ROWS)
importance = pd.concat(IMPORTANCE, axis=1)
results.to_csv(os.path.join(RESULTS, "results.csv"), index=False)
per_count.to_csv(os.path.join(RESULTS, "per_count_recall.csv"), index=False)
importance.to_csv(os.path.join(RESULTS, "gbm_feature_importance.csv"))

cols = ["setting", "model", "task", "calib", "mae", "exact", "within1", "acc4", "f1_4", "n"]
ens = results[results["level"] == "view_ensemble"]
best = (ens.sort_values("f1_4", ascending=False)
        .groupby(["setting", "model"]).head(1)
        .sort_values(["setting", "f1_4"], ascending=[True, False]))

# what ONE ESP32 link sees: models that use a single link, scored per link
single_models = ["gbm_base10", "gbm_single_link", "cnn_single", "cnn_single_int8"]
one_link = results[(results["level"] == "single_view") & results["model"].isin(single_models)]
one_link_best = (one_link.sort_values("f1_4", ascending=False)
                 .groupby(["setting", "model"]).head(1)
                 .sort_values(["setting", "f1_4"], ascending=[True, False]))

# ---- plot 1: 0/1/2/3+ macro F1 per model and setting ----
pivot = best.pivot(index="model", columns="setting", values="f1_4")[list(SETTINGS)]
ax = pivot.sort_values("zero_shot").plot.barh(figsize=(9, 6))
ax.set_xlabel("macro F1, 0 / 1 / 2 / 3+ (view ensemble, best task + calib)")
ax.set_xlim(0, 1)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS, "f1_4_by_model.png"), dpi=120)
plt.show()

# ---- plot 2: per-count recall of the best 6-class model per setting ----
fig, axes = plt.subplots(1, len(SETTINGS), figsize=(6 * len(SETTINGS), 4))
for ax, setting in zip(axes, SETTINGS):
    top = (ens[(ens["setting"] == setting) & (ens["task"] != "clf4")]
           .sort_values("mae").drop_duplicates("model").head(4))
    for _, r in top.iterrows():
        h = per_count[(per_count["setting"] == setting) & (per_count["model"] == r["model"])
                      & (per_count["task"] == r["task"]) & (per_count["calib"] == r["calib"])]
        ax.plot(h["count"], h["recall"], marker="o", label=f"{r['model']}/{r['task']}/{r['calib']}")
    ax.set_title(f"{setting}: per-count recall (4 lowest-MAE models)")
    ax.set_xlabel("people")
    ax.set_ylim(0, 1)
    ax.legend(fontsize=6)
fig.tight_layout()
fig.savefig(os.path.join(RESULTS, "per_count_recall.png"), dpi=120)
plt.show()

imp = importance.copy()
imp.columns = [c.split("|")[0] for c in imp.columns]
imp = imp.T.groupby(level=0).mean().T
imp = (imp / imp.sum()).round(3)

lines = ["# WiMANS - people counting v3 (5 GHz, unseen room)", "",
         f"Generated {time.strftime('%Y-%m-%d %H:%M')}; 3-s windows, {FS} Hz, 52 tones, 9 links per sample.", "",
         "## Best per model and setting (all 9 links averaged; best task + calibration by 0/1/2/3+ F1)", "",
         best[cols].round(3).to_markdown(index=False), "",
         "## One ESP32 link (single-link models, scored per link)", "",
         one_link_best[cols].round(3).to_markdown(index=False), "",
         "## Deployment (final models trained on all 5 GHz samples)", "",
         pd.DataFrame(DEPLOY).T.to_markdown(), "",
         "## LightGBM feature importance (share of gain, zero-shot)", "", imp.to_markdown(), "",
         "## Caveats", "",
         "- Intel 5300 30 grouped subcarriers were interpolated to 52 tones; no ESP32 hardware involved.",
         "- group_z calibration uses all unlabelled samples of the test room (slightly optimistic).",
         "- cal_* settings add the first 30 s of one session per count (cal_empty: count 0 only); those sessions are never tested.",
         "- int8 rows quantise weights only; activations stay float. BatchNorm would be folded into the convs on device.",
         "- multi-link models need all 9 links of a sample at once (several receivers).",
         "- Packet times are assumed uniform at 1 kHz (amp files have no timestamps)."]
with open(os.path.join(RESULTS, "WIMANS_RESULTS.md"), "w") as f:
    f.write(chr(10).join(lines))
display(best[cols].round(3))
display(one_link_best[cols].round(3))